# Expanded catalog: convert `product-models.json` to the catalog schema

`data/product-models.json` is a newer export with **349 product models** (the original `data/catalog.json` has 208 products). It uses a flat schema, so the preprocessing in `1_data_overview_and_preprocessing.ipynb` cannot read it directly.

This notebook rewrites it into the same schema as `data/catalog.json` and saves it as `data/expanded_catalog.json`. After that, notebook 1 can run on the expanded file by changing one path.

Mapping (flat model on the left, catalog schema on the right):

| `product-models.json` | `catalog.json` schema |
|---|---|
| `id` | `id` |
| `category` | `data.classification.product_type` |
| `gender`, `universe` | `data.classification.gender`, `.universe` |
| `introSeason` | `data.classification.introduction_season` |
| `lifecycle` | `data.status.lifecycle_3pr` |
| `name` | `data.content[en-GB].name` |
| `description` | `data.content[en-GB].description_html` |
| `facts[].label` / `facts[].value` | `data.content[en-GB].feature_titles` / `.features` (same order, so the pairing is kept) |
| `colourways[]` | `variants[]` (colour name and code, SKUs, media) |
| colourway `lifecycle` / `introSeason` that differ from the model | `variants[].overrides.status` / `.classification` |

The new export has **no** prices, trade data, composition, care text, swatches, logistics or non-English text. Those fields are kept in the schema but left empty (`null`, `""` or `{}`), never invented.

In [ ]:
import json
from collections import Counter
from pathlib import Path

data_dir = Path("data")
if not (data_dir / "catalog.json").is_file():
    data_dir = Path("CTFDS-Project/data")

source_path = data_dir / "product-models.json"
reference_path = data_dir / "catalog.json"
output_path = data_dir / "expanded_catalog.json"

models = json.loads(source_path.read_text(encoding="utf-8"))
reference = json.loads(reference_path.read_text(encoding="utf-8"))
print(f"Source models: {len(models)} | reference products: {len(reference['products'])}")
print("Model keys:", sorted({key for model in models for key in model}))
print("Models without facts:", sum(not model.get('facts') for model in models))
print("Models without description:", sum(not model.get('description') for model in models))

## Convert

Only fields that exist in the source are filled. A colourway whose lifecycle or intro season equals the model's gets no override, which matches how the original catalog only stores overrides when a variant differs.

In [ ]:
LOCALE = reference.get("default_locale", "en-GB")


def convert_sku(sku):
    stock = {
        "quantity": sku.get("stockQuantity"),
        "label": sku.get("stockLabel"),
        "available": sku.get("inStock"),
    }
    if sku.get("nextReplenishment"):
        stock["next_delivery"] = sku["nextReplenishment"]
    return {
        "id": sku["id"],
        "gtin13": sku.get("ean"),
        "size": {"alpha": sku.get("sizeLabel"), "code": sku.get("sizeCode")},
        "stock": stock,
    }


def convert_colourway(colourway, model):
    variant = {
        "id": colourway["id"],
        "color": {"name": colourway.get("colourName"), "code": colourway.get("colourCode"), "swatch": None},
        "media": {"product_images": [colourway["mediaUrl"]]} if colourway.get("mediaUrl") else {},
        "skus": [convert_sku(sku) for sku in colourway.get("sizeSkus") or []],
    }
    overrides = {}
    if colourway.get("lifecycle") and colourway["lifecycle"] != model["lifecycle"]:
        overrides["status"] = {"lifecycle_3pr": colourway["lifecycle"]}
    if colourway.get("introSeason") and colourway["introSeason"] != model["introSeason"]:
        overrides["classification"] = {"introduction_season": colourway["introSeason"]}
    if overrides:
        variant["overrides"] = overrides
    return variant


def convert_model(model):
    facts = model.get("facts") or []
    content = {
        "name": model["name"],
        "composition": "",
        "feature_titles": [fact["label"] for fact in facts],
        "care": "",
    }
    if facts:
        content["features"] = [fact["value"] for fact in facts]
    if model.get("description"):
        content["description_html"] = model["description"]

    colourways = model.get("colourways") or []
    first_media = next((cw["mediaUrl"] for cw in colourways if cw.get("mediaUrl")), None)
    return {
        "id": model["id"],
        "data": {
            "status": {"lifecycle_3pr": model["lifecycle"], "preorder_season": None, "content": None},
            "classification": {
                "product_type": model["category"],
                "gender": model["gender"],
                "universe": model["universe"],
                "introduction_season": model["introSeason"],
            },
            "trade": {"tariff_number": None, "tariff_type": None, "country_of_origin": None},
            "prices": {},
            "content": {LOCALE: content},
        },
        "media": {"main": first_media} if first_media else {},
        "variants": [convert_colourway(colourway, model) for colourway in colourways],
    }


expanded = {
    "schema_version": reference["schema_version"],
    "default_locale": LOCALE,
    "locales": [LOCALE],  # the export only contains English text
    "currencies": [],     # the export has no prices
    "inheritance": reference["inheritance"],
    "products": [convert_model(model) for model in models],
}
print(f"Converted {len(expanded['products'])} products, {sum(len(p['variants']) for p in expanded['products'])} variants")

## Check against the original schema and save

In [ ]:
assert list(expanded) == list(reference), (list(expanded), list(reference))
ref_product = reference["products"][0]
for product in expanded["products"]:
    assert set(product) == set(ref_product), product["id"]
    assert set(product["data"]) == set(ref_product["data"]), product["id"]
    assert set(product["data"]["classification"]) == set(ref_product["data"]["classification"]), product["id"]
    content = product["data"]["content"][LOCALE]
    features = content.get("features")
    assert features is None or len(features) == len(content["feature_titles"]), product["id"]
assert len({p["id"] for p in expanded["products"]}) == len(expanded["products"]), "duplicate ids"

output_path.write_text(json.dumps(expanded, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)")
print("Top-level keys match catalog.json:", list(expanded) == list(reference))

## Compare with the original catalog

The original file's 208 products should all be present. Feature titles that differ are listed by kind: pure capitalisation or punctuation changes versus real wording changes.

In [ ]:
import html
import re

original = {product["id"]: product for product in reference["products"]}
converted = {product["id"]: product for product in expanded["products"]}
print(f"In original: {len(original)} | in expanded: {len(converted)}")
print(f"Original ids missing from expanded: {sorted(set(original) - set(converted))}")
print(f"New products: {len(set(converted) - set(original))}")


def titles(product):
    content = (product["data"].get("content") or {}).get(LOCALE) or {}
    return [html.unescape(t).strip() for t in content.get("feature_titles") or []]


def normalise(title):
    return re.sub(r"[^a-z0-9]+", " ", title.lower()).strip()


identical = cosmetic = changed = 0
changed_examples = []
for product_id, product in original.items():
    if product_id not in converted:
        continue
    old, new = titles(product), titles(converted[product_id])
    if old == new:
        identical += 1
    elif [normalise(t) for t in old] == [normalise(t) for t in new]:
        cosmetic += 1
    else:
        changed += 1
        changed_examples.append((product_id, old, new))
print(f"Feature titles identical: {identical} | only case/punctuation differs: {cosmetic} | different: {changed}")
for product_id, old, new in changed_examples[:5]:
    print(f"\n{product_id}\n  original: {old}\n  expanded: {new}")

counts = Counter(t for p in expanded["products"] for t in set(titles(p)))
singles = sum(count == 1 for count in counts.values())
print(f"\nExpanded catalog: {len(counts)} distinct feature titles, {singles} appear on exactly one product")
old_counts = Counter(t for p in reference["products"] for t in set(titles(p)))
print(f"Original catalog: {len(old_counts)} distinct feature titles, {sum(c == 1 for c in old_counts.values())} appear on exactly one product")